# VinDr-Mammo Resumable Downloader

## ✨ Key Feature: RESUME DOWNLOADS

**If download gets interrupted, just run the resume cell to continue!**

Features:
- ✅ **Auto-resume:** Continues from where it left off
- ✅ **Progress saved:** Every 50 files to Google Drive
- ✅ **File verification:** Checks file size before skipping
- ✅ **Crash-proof:** Works even if Colab disconnects
- ✅ **Simple:** Just call `downloader.resume()`

---

In [ ]:
import os
import json
import subprocess
from pathlib import Path
from typing import List, Dict
import getpass
import pandas as pd
import numpy as np
from tqdm import tqdm
import time
from datetime import datetime

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted")

## Step 2: Install aria2c

In [ ]:
!apt-get update -qq
!apt-get install -y -qq aria2
print("✅ aria2c installed")

## Step 3: Resumable Downloader Class

In [ ]:
class VinDrMammoResumableDownloader:
    """
    Resumable downloader with automatic progress saving.
    
    Key features:
    - Automatically resumes from last saved position
    - Verifies file integrity before skipping
    - Saves progress every 10 files
    - Works even if Colab disconnects
    - Can retry failed downloads
    """
    
    def __init__(self, gdrive_path='/content/drive/MyDrive/vindr-mammo-stratified', 
                 max_total_files=1000, connections_per_file=5, concurrent_downloads=3):
        self.base_dir = Path(gdrive_path)
        self.base_url = "https://physionet.org/files/vindr-mammo/1.0.0"
        
        self.username = None
        self.password = None
        
        self.max_total_files = max_total_files
        self.connections_per_file = connections_per_file
        self.concurrent_downloads = concurrent_downloads
        self.random_seed = 42
        
        # Create directories
        self.base_dir.mkdir(parents=True, exist_ok=True)
        (self.base_dir / 'images').mkdir(exist_ok=True)
        (self.base_dir / 'metadata').mkdir(exist_ok=True)
        
        self.progress_file = self.base_dir / 'download_progress.json'
        self.selection_file = self.base_dir / 'metadata' / 'selected_files.csv'
        self.credentials_file = self.base_dir / 'credentials.json'
        
        # Load saved credentials if they exist
        self._load_credentials()
        
        print(f"✅ Initialized RESUMABLE downloader at {self.base_dir}")
        print(f"   Target: ~{max_total_files} files")
        print(f"   Progress saved to Google Drive (survives disconnections)")
    
    def setup_credentials(self, username: str = None, password: str = None) -> bool:
        """Setup and save credentials."""
        if not username:
            print("\n🔐 PhysioNet Credentials")
            username = input("Username: ").strip()
            password = getpass.getpass("Password: ")
        
        self.username = username
        self.password = password
        
        # Save credentials for resume
        with open(self.credentials_file, 'w') as f:
            json.dump({'username': username, 'password': password}, f)
        
        print("✅ Credentials saved (for auto-resume)")
        return True
    
    def _load_credentials(self):
        """Load saved credentials."""
        if self.credentials_file.exists():
            try:
                with open(self.credentials_file, 'r') as f:
                    creds = json.load(f)
                    self.username = creds.get('username')
                    self.password = creds.get('password')
                    if self.username and self.password:
                        print("   ✅ Loaded saved credentials")
            except:
                pass
    
    def download_metadata(self) -> bool:
        """Download metadata."""
        print("\n📊 Downloading Metadata")
        print("=" * 70)
        
        metadata_dir = self.base_dir / 'metadata'
        csv_file = 'breast-level_annotations.csv'
        output_file = metadata_dir / csv_file
        
        if output_file.exists():
            print(f"  ✅ {csv_file} already exists")
            return True
        
        print(f"  📥 Downloading {csv_file}...")
        
        url = f"{self.base_url}/{csv_file}"
        cmd = [
            'aria2c',
            f'--http-user={self.username}',
            f'--http-passwd={self.password}',
            '-d', str(metadata_dir),
            '-o', csv_file,
            '--max-tries=3',
            '-x', '5',
            url
        ]
        
        try:
            result = subprocess.run(cmd, capture_output=True, timeout=90)
            if result.returncode == 0 and output_file.exists():
                print(f"  ✅ Downloaded ({output_file.stat().st_size / (1024 * 1024):.2f} MB)")
                return True
            else:
                print(f"  ❌ Failed")
                return False
        except Exception as e:
            print(f"  ❌ Error: {e}")
            return False
    
    def perform_stratified_selection(self) -> pd.DataFrame:
        """Perform stratified selection."""
        print("\n🎯 Stratified Selection")
        print(f"   Target: ~{self.max_total_files} files")
        print("=" * 70)
        
        csv_file = self.base_dir / 'metadata' / 'breast-level_annotations.csv'
        
        if not csv_file.exists():
            print("❌ Metadata not found")
            return None
        
        df = pd.read_csv(csv_file)
        df['birads_numeric'] = df['breast_birads'].str.extract(r'(\d+)')[0].astype(float)
        df_filtered = df[df['birads_numeric'] != 3].copy()
        df_filtered['label'] = df_filtered['birads_numeric'].apply(
            lambda x: 1 if x in [4, 5, 6] else 0
        )
        
        malignant_df = df_filtered[df_filtered['label'] == 1]
        benign_df = df_filtered[df_filtered['label'] == 0]
        
        np.random.seed(self.random_seed)
        target_malignant = int(self.max_total_files * 0.25)
        target_benign = int(self.max_total_files * 0.75)
        
        malignant_patient_sizes = malignant_df.groupby('study_id').size().reset_index(name='image_count')
        benign_patient_sizes = benign_df.groupby('study_id').size().reset_index(name='image_count')
        
        malignant_patient_sizes = malignant_patient_sizes.sample(frac=1, random_state=self.random_seed).reset_index(drop=True)
        benign_patient_sizes = benign_patient_sizes.sample(frac=1, random_state=self.random_seed).reset_index(drop=True)
        
        # Select patients
        selected_malignant_patients = []
        malignant_count = 0
        for _, row in malignant_patient_sizes.iterrows():
            if malignant_count >= target_malignant:
                break
            selected_malignant_patients.append(row['study_id'])
            malignant_count += row['image_count']
        
        selected_benign_patients = []
        benign_count = 0
        for _, row in benign_patient_sizes.iterrows():
            if benign_count >= target_benign:
                break
            selected_benign_patients.append(row['study_id'])
            benign_count += row['image_count']
        
        malignant_selected = malignant_df[malignant_df['study_id'].isin(selected_malignant_patients)]
        benign_selected = benign_df[benign_df['study_id'].isin(selected_benign_patients)]
        selected_df = pd.concat([malignant_selected, benign_selected], ignore_index=True)
        
        print(f"\n  ✅ Selected {len(selected_df)} images")
        print(f"     Malignant: {len(malignant_selected)} ({len(selected_malignant_patients)} patients)")
        print(f"     Benign: {len(benign_selected)} ({len(selected_benign_patients)} patients)")
        
        selected_df.to_csv(self.selection_file, index=False)
        print(f"\n  💾 Saved to: {self.selection_file}")
        
        return selected_df
    
    def check_progress(self):
        """Check current download progress."""
        print("\n" + "=" * 70)
        print("📊 DOWNLOAD PROGRESS")
        print("=" * 70)
        
        if not self.selection_file.exists():
            print("❌ No selection file found. Run perform_stratified_selection() first.")
            return
        
        selected_df = pd.read_csv(self.selection_file)
        total_files = len(selected_df)
        
        progress = self._load_progress()
        downloaded_files = set(progress.get('downloaded_files', []))
        
        # Verify downloaded files exist
        verified_downloaded = []
        for file_path in downloaded_files:
            full_path = self.base_dir / file_path
            if full_path.exists() and full_path.stat().st_size > 0:
                verified_downloaded.append(file_path)
        
        remaining = total_files - len(verified_downloaded)
        progress_pct = (len(verified_downloaded) / total_files * 100) if total_files > 0 else 0
        
        print(f"\n📋 Target files: {total_files}")
        print(f"✅ Downloaded: {len(verified_downloaded)} ({progress_pct:.1f}%)")
        print(f"⏳ Remaining: {remaining}")
        print(f"❌ Failed: {len(progress.get('failed_files', []))}")
        
        if progress.get('last_update'):
            print(f"\n⏰ Last update: {progress['last_update']}")
        
        if remaining > 0:
            est_time_hours = (remaining * 25) / 3600  # Assume 25 sec/file
            print(f"\n⏱️  Estimated time remaining: ~{est_time_hours:.1f} hours")
            print(f"\n💡 To resume: Run the cell that calls downloader.resume()")
        else:
            print(f"\n🎉 All files downloaded!")
        
        print("=" * 70 + "\n")
    
    def resume(self, batch_size=50):
        """
        Resume download from where it left off.
        
        This is the main method to call if download gets interrupted!
        """
        print("\n" + "=" * 70)
        print("🔄 RESUMING DOWNLOAD")
        print("=" * 70)
        
        # Check credentials
        if not self.username or not self.password:
            print("❌ No credentials found. Please run setup_credentials() first.")
            return False
        
        # Check selection file
        if not self.selection_file.exists():
            print("❌ No selection file found. Run perform_stratified_selection() first.")
            return False
        
        selected_df = pd.read_csv(self.selection_file)
        print(f"\n✅ Loaded selection: {len(selected_df)} files total")
        
        # Load progress
        progress = self._load_progress()
        downloaded_files = set(progress.get('downloaded_files', []))
        
        # Verify and count already downloaded
        verified_count = 0
        for file_path in list(downloaded_files):
            full_path = self.base_dir / file_path
            if full_path.exists() and full_path.stat().st_size > 0:
                verified_count += 1
            else:
                # File missing or corrupted, remove from progress
                downloaded_files.discard(file_path)
        
        print(f"✅ Already downloaded: {verified_count} files (verified)")
        
        # Prepare remaining files
        files_to_download = []
        for idx, row in selected_df.iterrows():
            study_id = row['study_id']
            image_id = row['image_id']
            image_path = f"images/{study_id}/{image_id}.dicom"
            
            if image_path not in downloaded_files:
                output_file = self.base_dir / image_path
                output_file.parent.mkdir(parents=True, exist_ok=True)
                url = f"{self.base_url}/{image_path}"
                files_to_download.append((url, output_file, image_path))
        
        if not files_to_download:
            print("\n🎉 All files already downloaded!")
            return True
        
        print(f"⏳ Remaining: {len(files_to_download)} files")
        print(f"\n🚀 Starting download...\n")
        
        # Download with progress
        return self._download_batch(files_to_download, progress, batch_size)
    
    def retry_failed(self, batch_size=50, max_retries=3):
        """
        Retry downloading only the failed files.
        
        Args:
            batch_size: Files per batch
            max_retries: Number of retry attempts per file
        
        Returns:
            True if all succeeded
        """
        print("\n" + "=" * 70)
        print("🔄 RETRYING FAILED DOWNLOADS")
        print("=" * 70)
        
        # Load progress
        progress = self._load_progress()
        failed_files = progress.get('failed_files', [])
        
        if not failed_files:
            print("\n✅ No failed files to retry!")
            return True
        
        print(f"\n❌ Found {len(failed_files)} failed files")
        print(f"🔄 Will retry with {max_retries} attempts per file\n")
        
        # Prepare retry list
        files_to_retry = []
        for failed_path in failed_files:
            output_file = self.base_dir / failed_path
            output_file.parent.mkdir(parents=True, exist_ok=True)
            url = f"{self.base_url}/{failed_path}"
            
            files_to_retry.append((url, output_file, failed_path))
        
        print(f"📥 Retrying {len(files_to_retry)} files...\n")
        
        # Clear failed list before retry
        progress['failed_files'] = []
        
        # Download with retries
        success_count = 0
        still_failed = []
        
        # Process in batches
        num_batches = (len(files_to_retry) + batch_size - 1) // batch_size
        
        for batch_idx in range(num_batches):
            start_idx = batch_idx * batch_size
            end_idx = min(start_idx + batch_size, len(files_to_retry))
            batch = files_to_retry[start_idx:end_idx]
            
            print(f"📦 Retry Batch {batch_idx + 1}/{num_batches}: {len(batch)} files")
            
            # Try each file multiple times
            for url, output_file, image_path in batch:
                downloaded = False
                
                for attempt in range(max_retries):
                    # Download single file
                    cmd = [
                        'aria2c',
                        f'--http-user={self.username}',
                        f'--http-passwd={self.password}',
                        '-d', str(output_file.parent),
                        '-o', output_file.name,
                        '-x', str(self.connections_per_file),
                        '--max-tries=2',
                        '--retry-wait=3',
                        '--timeout=90',
                        '--allow-overwrite=true',
                        url
                    ]
                    
                    try:
                        result = subprocess.run(cmd, capture_output=True, timeout=120)
                        
                        if result.returncode == 0 and output_file.exists() and output_file.stat().st_size > 0:
                            # Success!
                            success_count += 1
                            if image_path not in progress['downloaded_files']:
                                progress['downloaded_files'].append(image_path)
                            downloaded = True
                            break
                        else:
                            # Failed, wait before retry
                            if attempt < max_retries - 1:
                                time.sleep(5)
                    
                    except Exception as e:
                        if attempt < max_retries - 1:
                            time.sleep(5)
                
                if not downloaded:
                    still_failed.append(image_path)
            
            # Save progress after each batch
            progress['failed_files'] = still_failed
            self._save_progress(progress)
            
            print(f"   ✅ Batch complete: {success_count}/{len(files_to_retry)} total recovered\n")
        
        print("=" * 70)
        print(f"✅ Retry Complete!")
        print(f"   Recovered: {success_count}")
        print(f"   Still failed: {len(still_failed)}")
        print(f"   Success rate: {success_count / len(files_to_retry) * 100:.1f}%")
        print("=" * 70 + "\n")
        
        if still_failed:
            print(f"⚠️  {len(still_failed)} files still failed after {max_retries} retries")
            print(f"💡 You can run retry_failed() again to try once more")
            print(f"💡 Or check if there's a network/PhysioNet issue\n")
        
        return len(still_failed) == 0
    
    def _download_batch(self, files_to_download, progress, batch_size):
        """Internal method to download files in batches."""
        total_success = 0
        total_failed = 0
        num_batches = (len(files_to_download) + batch_size - 1) // batch_size
        
        for batch_idx in range(num_batches):
            start_idx = batch_idx * batch_size
            end_idx = min(start_idx + batch_size, len(files_to_download))
            batch = files_to_download[start_idx:end_idx]
            
            print(f"📦 Batch {batch_idx + 1}/{num_batches}: {len(batch)} files")
            
            # Create aria2c input file
            input_file = self.base_dir / f'aria2c_batch_{batch_idx}.txt'
            
            with open(input_file, 'w') as f:
                for url, output_file, _ in batch:
                    f.write(f"{url}\n")
                    f.write(f"  out={output_file}\n")
                    f.write(f"  http-user={self.username}\n")
                    f.write(f"  http-passwd={self.password}\n")
            
            # Run aria2c
            cmd = [
                'aria2c',
                '-i', str(input_file),
                f'-j{self.concurrent_downloads}',
                f'-x{self.connections_per_file}',
                '--max-tries=3',
                '--retry-wait=2',
                '--timeout=60',
                '--allow-overwrite=true',
                '--auto-file-renaming=false'
            ]
            
            try:
                subprocess.run(cmd, capture_output=False, timeout=batch_size * 60)
                
                # Verify downloads
                batch_success = 0
                batch_failed = 0
                
                for url, output_file, image_path in batch:
                    if output_file.exists() and output_file.stat().st_size > 0:
                        batch_success += 1
                        if image_path not in progress['downloaded_files']:
                            progress['downloaded_files'].append(image_path)
                    else:
                        batch_failed += 1
                        if image_path not in progress.get('failed_files', []):
                            progress.setdefault('failed_files', []).append(image_path)
                
                total_success += batch_success
                total_failed += batch_failed
                
                print(f"   ✅ {batch_success} success, ❌ {batch_failed} failed\n")
                
                # Save progress frequently
                self._save_progress(progress)
                
                input_file.unlink()
                
            except Exception as e:
                print(f"   ❌ Batch error: {e}\n")
                if input_file.exists():
                    input_file.unlink()
        
        print("=" * 70)
        print(f"✅ Batch Download Complete")
        print(f"   New downloads: {total_success}")
        print(f"   Failed: {total_failed}")
        print(f"   Total downloaded: {len(progress['downloaded_files'])}")
        print("=" * 70 + "\n")
        
        return total_failed == 0
    
    def _load_progress(self) -> Dict:
        """Load progress."""
        if self.progress_file.exists():
            try:
                with open(self.progress_file, 'r') as f:
                    return json.load(f)
            except:
                pass
        return {'downloaded_files': [], 'failed_files': []}
    
    def _save_progress(self, progress: Dict):
        """Save progress to Google Drive."""
        progress['last_update'] = datetime.now().isoformat()
        with open(self.progress_file, 'w') as f:
            json.dump(progress, f, indent=2)

## 🚀 First Time Setup (Run Once)

In [ ]:
# Initialize downloader
downloader = VinDrMammoResumableDownloader(
    gdrive_path='/content/drive/MyDrive/vindr-mammo-stratified',
    max_total_files=1000,
    connections_per_file=5,
    concurrent_downloads=3
)

In [ ]:
# Setup credentials (saved for auto-resume)
downloader.setup_credentials()

In [ ]:
# Download metadata
downloader.download_metadata()

In [ ]:
# Perform stratified selection
selected_df = downloader.perform_stratified_selection()

## 📥 Start/Resume Download (Run This Cell)

**This is the main download cell. Run it to:**
- Start download for the first time
- Resume if interrupted
- Continue after Colab disconnect

In [ ]:
# Start or resume download
downloader.resume(batch_size=50)

## 📊 Check Progress (Optional)

Run this anytime to see current progress

In [ ]:
# Check current progress
downloader.check_progress()

## 🔄 Retry Failed Downloads

If some files failed, use this to retry them with multiple attempts

In [ ]:
# Retry failed downloads with multiple attempts
# This will try each failed file 3 times before giving up
downloader.retry_failed(batch_size=50, max_retries=3)

## 🔍 Diagnose Failed Downloads

**Run this to understand WHY files are failing**

This will test a sample of failed files to categorize error types:
- 403 Forbidden (file doesn't exist on server - NOT recoverable)
- Timeout errors (may be recoverable)
- Network errors (may be recoverable)
- Other errors

---

In [ ]:
# Analyze distribution of downloaded files
def analyze_downloaded_distribution(downloader):
    """Check if downloaded files maintain proper stratification."""
    
    print("=" * 80)
    print("📊 DOWNLOADED FILES DISTRIBUTION ANALYSIS")
    print("=" * 80)
    
    # Load progress
    progress = downloader._load_progress()
    downloaded_files = set(progress.get('downloaded_files', []))
    
    if not downloaded_files:
        print("\n❌ No downloaded files found")
        return None
    
    # Verify files exist
    verified_files = []
    for file_path in downloaded_files:
        full_path = downloader.base_dir / file_path
        if full_path.exists() and full_path.stat().st_size > 0:
            verified_files.append(file_path)
    
    print(f"\n✅ Total downloaded files: {len(verified_files)}")
    
    # Load selection to get labels
    if not downloader.selection_file.exists():
        print("❌ Selection file not found")
        return None
    
    selection_df = pd.read_csv(downloader.selection_file)
    
    # Create mapping of image_path to label
    selection_df['image_path'] = selection_df.apply(
        lambda row: f"images/{row['study_id']}/{row['image_id']}.dicom",
        axis=1
    )
    
    # Filter to only downloaded files
    downloaded_df = selection_df[selection_df['image_path'].isin(verified_files)].copy()
    
    if len(downloaded_df) == 0:
        print("❌ Could not match downloaded files with metadata")
        return None
    
    # Label distribution
    print("\n" + "=" * 80)
    print("🎯 LABEL DISTRIBUTION (Downloaded Files)")
    print("=" * 80)
    
    label_counts = downloaded_df['label'].value_counts()
    total = len(downloaded_df)
    
    malignant = label_counts.get(1, 0)
    benign = label_counts.get(0, 0)
    
    malignant_pct = (malignant / total * 100) if total > 0 else 0
    benign_pct = (benign / total * 100) if total > 0 else 0
    
    print(f"\n  Malignant (label=1): {malignant:4d} ({malignant_pct:5.1f}%)")
    print(f"  Benign (label=0):    {benign:4d} ({benign_pct:5.1f}%)")
    print(f"  Total:               {total:4d}")
    
    print(f"\n  Target distribution: 25% malignant / 75% benign")
    print(f"  Actual distribution: {malignant_pct:.1f}% malignant / {benign_pct:.1f}% benign")
    
    # Check deviation
    malignant_diff = abs(malignant_pct - 25.0)
    benign_diff = abs(benign_pct - 75.0)
    
    if malignant_diff < 5 and benign_diff < 5:
        print(f"\n  ✅ EXCELLENT: Within 5% of target distribution")
    elif malignant_diff < 10 and benign_diff < 10:
        print(f"\n  ✅ GOOD: Within 10% of target distribution")
    else:
        print(f"\n  ⚠️  WARNING: More than 10% deviation from target")
    
    # Patient-level analysis
    print("\n" + "=" * 80)
    print("👥 PATIENT-LEVEL ANALYSIS")
    print("=" * 80)
    
    unique_patients = downloaded_df['study_id'].nunique()
    malignant_patients = downloaded_df[downloaded_df['label'] == 1]['study_id'].nunique()
    benign_patients = downloaded_df[downloaded_df['label'] == 0]['study_id'].nunique()
    
    print(f"\n  Total unique patients: {unique_patients}")
    print(f"  Malignant patients:    {malignant_patients}")
    print(f"  Benign patients:       {benign_patients}")
    
    # Images per patient
    images_per_patient = downloaded_df.groupby('study_id').size()
    print(f"\n  Images per patient:")
    print(f"    Mean:   {images_per_patient.mean():.1f}")
    print(f"    Median: {images_per_patient.median():.1f}")
    print(f"    Min:    {images_per_patient.min()}")
    print(f"    Max:    {images_per_patient.max()}")
    
    # BI-RADS distribution
    print("\n" + "=" * 80)
    print("🏥 BI-RADS DISTRIBUTION")
    print("=" * 80)
    
    birads_counts = downloaded_df['breast_birads'].value_counts().sort_index()
    print("\n  BI-RADS Category | Count | Percentage")
    print("  " + "-" * 40)
    
    for birads, count in birads_counts.items():
        pct = (count / total * 100)
        label = "Malignant" if birads in ['BI-RADS 4', 'BI-RADS 5', 'BI-RADS 6'] else "Benign"
        print(f"  {birads:15s} | {count:5d} | {pct:5.1f}% ({label})")
    
    # Laterality distribution
    print("\n" + "=" * 80)
    print("🔄 LATERALITY DISTRIBUTION")
    print("=" * 80)
    
    if 'laterality' in downloaded_df.columns:
        laterality_counts = downloaded_df['laterality'].value_counts()
        print("\n  Side | Count | Percentage")
        print("  " + "-" * 30)
        
        for side, count in laterality_counts.items():
            pct = (count / total * 100)
            print(f"  {side:4s} | {count:5d} | {pct:5.1f}%")
    
    # View position distribution
    print("\n" + "=" * 80)
    print("📸 VIEW POSITION DISTRIBUTION")
    print("=" * 80)
    
    if 'view_position' in downloaded_df.columns:
        view_counts = downloaded_df['view_position'].value_counts()
        print("\n  View | Count | Percentage")
        print("  " + "-" * 30)
        
        for view, count in view_counts.items():
            pct = (count / total * 100)
            print(f"  {view:4s} | {count:5d} | {pct:5.1f}%")
    
    # Comparison with original selection
    print("\n" + "=" * 80)
    print("📊 COMPARISON: Original Selection vs Downloaded")
    print("=" * 80)
    
    orig_total = len(selection_df)
    orig_malignant = (selection_df['label'] == 1).sum()
    orig_benign = (selection_df['label'] == 0).sum()
    
    print(f"\n  Original Selection:")
    print(f"    Total:     {orig_total:4d}")
    print(f"    Malignant: {orig_malignant:4d} ({orig_malignant/orig_total*100:5.1f}%)")
    print(f"    Benign:    {orig_benign:4d} ({orig_benign/orig_total*100:5.1f}%)")
    
    print(f"\n  Downloaded:")
    print(f"    Total:     {total:4d} ({total/orig_total*100:5.1f}% of selection)")
    print(f"    Malignant: {malignant:4d} ({malignant_pct:5.1f}%)")
    print(f"    Benign:    {benign:4d} ({benign_pct:5.1f}%)")
    
    # Final verdict
    print("\n" + "=" * 80)
    print("✅ FINAL VERDICT")
    print("=" * 80)
    
    if malignant_diff < 5 and benign_diff < 5 and total >= 200:
        print("\n  ✅ EXCELLENT: Good representation with valid stratification")
        print(f"     - {total} files is sufficient for experimentation")
        print(f"     - Stratification is well-maintained")
        print(f"     - Proceed with model training")
    elif malignant_diff < 10 and benign_diff < 10 and total >= 100:
        print("\n  ✅ GOOD: Acceptable representation")
        print(f"     - {total} files is usable for development/testing")
        print(f"     - Stratification is reasonably maintained")
        print(f"     - May need more data for final model")
    else:
        print("\n  ⚠️  WARNING: Limited representation")
        print(f"     - {total} files may be too small")
        print(f"     - Stratification may be skewed")
        print(f"     - Consider requesting more data or alternative datasets")
    
    print("\n" + "=" * 80)
    
    return downloaded_df

# Run the analysis
downloaded_df = analyze_downloaded_distribution(downloader)

## 📊 Verify Dataset Distribution

**Check if downloaded files maintain stratification**

This analyzes:
- Malignant vs benign distribution (target: 25% / 75%)
- Patient-level distribution
- BI-RADS categories
- Laterality and view positions
- Comparison with original selection

---

In [ ]:
# Diagnostic: Analyze failed downloads
from collections import Counter

def diagnose_failed_downloads(downloader, sample_size=20):
    """
    Test a sample of failed files to categorize error types.
    
    Args:
        downloader: VinDrMammoResumableDownloader instance
        sample_size: Number of files to test (default: 20)
    """
    print("=" * 70)
    print("🔍 FAILED DOWNLOADS DIAGNOSTIC")
    print("=" * 70)
    
    # Load progress
    progress = downloader._load_progress()
    failed_files = progress.get('failed_files', [])
    
    if not failed_files:
        print("\n✅ No failed files to diagnose!")
        return
    
    print(f"\n📊 Total failed files: {len(failed_files)}")
    
    # Test sample
    actual_sample_size = min(sample_size, len(failed_files))
    print(f"🔍 Testing sample of {actual_sample_size} files...\n")
    
    error_counts = Counter()
    sample_indices = list(range(0, len(failed_files), max(1, len(failed_files) // actual_sample_size)))[:actual_sample_size]
    
    for i, idx in enumerate(sample_indices):
        file_path = failed_files[idx]
        print(f"[{i+1}/{actual_sample_size}] Testing: {file_path[:60]}...")
        
        # Test download
        url = f"{downloader.base_url}/{file_path}"
        output_path = downloader.base_dir / file_path
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        cmd = [
            'aria2c',
            f'--http-user={downloader.username}',
            f'--http-passwd={downloader.password}',
            '-d', str(output_path.parent),
            '-o', output_path.name,
            '-x', '5',
            '--max-tries=1',
            '--timeout=30',
            '--allow-overwrite=true',
            url
        ]
        
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
            output = result.stdout + result.stderr
            
            # Categorize error
            if "status=403" in output or "403 Forbidden" in output:
                error_type = '403_forbidden'
            elif "timed out" in output.lower() or "timeout" in output.lower():
                error_type = 'timeout'
            elif output_path.exists() and output_path.stat().st_size > 0:
                error_type = 'success'
                # Clean up test file
                output_path.unlink()
            elif "network" in output.lower() or "connection" in output.lower():
                error_type = 'network_error'
            else:
                error_type = 'unknown'
            
            error_counts[error_type] += 1
            print(f"  → {error_type}\n")
            
            # Small delay to avoid rate limiting
            if i < actual_sample_size - 1:
                time.sleep(2)
                
        except subprocess.TimeoutExpired:
            error_counts['timeout'] += 1
            print(f"  → timeout\n")
        except Exception as e:
            error_counts['unknown'] += 1
            print(f"  → unknown ({str(e)[:40]})\n")
    
    # Summary
    print("=" * 70)
    print("📊 DIAGNOSTIC SUMMARY")
    print("=" * 70)
    
    total_tested = sum(error_counts.values())
    
    for error_type, count in error_counts.most_common():
        percentage = (count / total_tested) * 100
        print(f"  {error_type:20s}: {count:3d} ({percentage:5.1f}%)")
    
    # Extrapolate
    print(f"\n📈 Extrapolated to all {len(failed_files)} failed files:")
    for error_type, count in error_counts.items():
        estimated = int((count / total_tested) * len(failed_files))
        print(f"  {error_type:20s}: ~{estimated} files")
    
    # Recommendations
    print("\n" + "=" * 70)
    print("💡 RECOMMENDATIONS")
    print("=" * 70)
    
    pct_403 = error_counts.get('403_forbidden', 0) / total_tested
    pct_timeout = error_counts.get('timeout', 0) / total_tested
    pct_success = error_counts.get('success', 0) / total_tested
    
    if pct_403 > 0.8:
        print("\n⚠️  MOST FILES HAVE 403 FORBIDDEN ERRORS")
        print("   → These files don't exist on PhysioNet's server")
        print("   → They CANNOT be recovered")
        print("   → This is a PhysioNet dataset issue, not a download problem")
        print(f"\n✅ RECOMMENDED ACTION:")
        print(f"   → Accept your current dataset size")
        downloaded_count = len(progress.get('downloaded_files', []))
        print(f"   → You have {downloaded_count} successfully downloaded files")
        print(f"   → This is {downloaded_count / 10:.1f}% of your target (1000 files)")
        
    elif pct_timeout > 0.5:
        print("\n⚠️  MANY TIMEOUT ERRORS")
        print("   → PhysioNet server may be slow/overloaded")
        print("   → Try downloading during off-peak hours (late night US time)")
        print("   → Or increase timeout values in the downloader")
        
    elif pct_success > 0.3:
        print("\n✅ SOME FILES CAN BE RECOVERED!")
        print("   → Try retry_failed() again")
        print("   → Or download during off-peak hours")
        print("   → Consider increasing timeout and retry values")
        
    else:
        print("\n❓ MIXED OR UNKNOWN ERRORS")
        print("   → May need manual investigation")
        print("   → Check PhysioNet access permissions")
        print("   → Verify internet connection stability")
    
    print("\n" + "=" * 70)

# Run diagnostic
diagnose_failed_downloads(downloader, sample_size=20)